In [ ]:
from pathlib import Path
import requests
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from zipfile import ZipFile


In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def download_if_needed(url, path):
    if not path.exists():
        print(f"Downloading {path.name}...")
        r = requests.get(url)
        r.raise_for_status()
        path.write_bytes(r.content)
    else:
        print(f"{path.name} already exists.")

def extract_if_needed(zip_path, extract_folder):
    if not extract_folder.exists():
        print(f"Extracting {zip_path.name}...")
        with ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(extract_folder)
    else:
        print(f"{extract_folder.name} already exists.")


loc_url = 'https://data.london.gov.uk/download/24rz6/77d9b319-931e-4090-bf8e-f578938bd352/LSOA2011%20AvPTAI2015.csv'
loc_path =  DATA_DIR / "location.csv"
crime_url = 'https://data.london.gov.uk/download/exy3m/vm7/MPS%20LSOA%20Level%20Crime%20(Historical).csv'
crime_path = DATA_DIR / "crime.csv"
deprivation_url = "https://assets.publishing.service.gov.uk/media/5dc407b440f0b6379a7acc8d/File_7_-_All_IoD2019_Scores__Ranks__Deciles_and_Population_Denominators_3.csv"
deprivation_path = DATA_DIR / "deprivation.csv" 
house_url = "https://www.ons.gov.uk/file?uri=/peoplepopulationandcommunity/housing/datasets/medianpricepaidbylowerlayersuperoutputareahpssadataset46/current/hpssadataset46medianpricepaidforresidentialpropertiesbylsoa.zip"
house_zip = DATA_DIR / "house_prices.zip"
house_path = DATA_DIR / "house_prices"
download_if_needed(loc_url,loc_path)
download_if_needed(crime_url,crime_path)
download_if_needed(deprivation_url,deprivation_path)
download_if_needed(house_url, house_zip)
extract_if_needed(house_zip, house_path)

In [ ]:
excel_file = next(house_path.glob("*.xls"))
loc_df = pd.read_csv('data/location.csv')
crime_df = pd.read_csv('data/crime.csv')
IoD_df = pd.read_csv('data/deprivation.csv')
house_df = pd.read_excel(
    excel_file,
    sheet_name="1a",
    header=5
)

In [ ]:
print(loc_df.columns)
print(crime_df.columns)
print(IoD_df.columns)
print(house_df.columns)

In [ ]:
loc_df = loc_df[['LSOA2011', 'AvPTAI2015', 'PTAL']]
crime_df = crime_df[['LSOA Code', 'LSOA Name', 'Borough', 'Group', 'SubGroup','202201', '202202', '202203', '202204', '202205', '202206',
       '202207', '202208', '202209', '202210', '202211', '202212']]
IoD_df = IoD_df[['LSOA code (2011)', 'LSOA name (2011)',
       'Local Authority District code (2019)',
       'Local Authority District name (2019)',
       'Index of Multiple Deprivation (IMD) Score']]
house_df = house_df[['Local authority code', 'Local authority name', 'LSOA code',
       'LSOA name','Year ending Mar 2022',
       'Year ending Jun 2022', 'Year ending Sep 2022', 'Year ending Dec 2022']]


In [ ]:
print(loc_df.head())
print(crime_df.head())
print(IoD_df.head())
print(house_df.head())

In [ ]:
house_london = house_df[house_df["Local authority code"].astype(str).str.startswith("E09", na=False)].copy()
price_columns = [
    "Year ending Mar 2022",
    "Year ending Jun 2022",
    "Year ending Sep 2022",
    "Year ending Dec 2022"
]

# Convert to numbers in case Excel imported any values as text
house_london[price_columns] = house_london[price_columns].apply(pd.to_numeric,errors="coerce")

house_london["average_price_2022"] = (house_london[price_columns].mean(axis=1))

house_london = house_london.drop(columns = price_columns)

print(house_london.head(20))

In [ ]:
crime_months = [
    "202201", "202202", "202203", "202204",
    "202205", "202206", "202207", "202208",
    "202209", "202210", "202211", "202212"
]

crime_df[crime_months] = crime_df[crime_months].apply(pd.to_numeric,errors="coerce").fillna(0)

crime_df["crime_total_2022"] = crime_df[crime_months].sum(axis=1)
total_crime_df = crime_df.drop(columns = crime_months).copy()



In [ ]:
crime_by_lsoa = (
    total_crime_df
    .pivot_table(
        index=["LSOA Code", "LSOA Name", "Borough"],
        columns="Group",
        values="crime_total_2022",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

# Remove the name added above the pivoted columns
crime_by_lsoa.columns.name = None

# Add total crime across all crime-category columns
crime_category_columns = [
    col for col in crime_by_lsoa.columns
    if col not in ["LSOA Code", "LSOA Name", "Borough"]
]

crime_by_lsoa["total_crime_2022"] = (
    crime_by_lsoa[crime_category_columns].sum(axis=1)
)

In [ ]:
look_up_url = 'https://open-geography-portalx-ons.hub.arcgis.com/api/download/v1/items/b684a0dbf786473f9563ec0616da2f8b/csv?layers=0'
look_up_path = DATA_DIR / "lookup.csv" 
download_if_needed(look_up_url,look_up_path)
lookup_df = pd.read_csv('data/lookup.csv')
lookup_df = lookup_df[['LSOA11CD','LSOA21CD']]


In [ ]:
crime_2011 = crime_by_lsoa.merge(
    lookup_df,
    left_on="LSOA Code",
    right_on="LSOA21CD",
    how="inner"
)

In [ ]:
house_crime = house_london.merge(
    crime_2011,
    left_on="LSOA code",
    right_on="LSOA11CD",
    how="left",
    validate="one_to_one"
)



house_crime = house_crime.drop(
    columns=["LSOA Code", "LSOA Name", "LSOA11CD","LSOA21CD", "Borough"],
    errors="ignore"
)
house_crime.head()

In [ ]:
print(f"House price LSOAs: {len(house_london)}")
print(f"Crime LSOAs after lookup: {len(crime_2011)}")
print(f"Merged dataset: {len(house_crime)}")
print(f"Unmatched house-price LSOAs: {house_crime['total_crime_2022'].isna().sum()}")


missing = house_crime[
    house_crime["total_crime_2022"].isna()
]

missing["Local authority name"].value_counts()

In [ ]:
deprivation_transport = IoD_df.merge(
    loc_df,
    left_on="LSOA code (2011)",
    right_on="LSOA2011",
    how="left",
    validate="one_to_one"
)

# Drop the duplicate transport LSOA code
deprivation_transport = deprivation_transport.drop(columns=['LSOA name (2011)','LSOA2011'])
deprivation_transport = deprivation_transport.rename(
    columns={
        'Local Authority District code (2019)' : 'LA code',
        'Local Authority District name (2019)' : 'LA name',
        'Index of Multiple Deprivation (IMD) Score': 'IMD score',
        "LSOA code (2011)": "LSOA code"
    }
)
deprivation_transport.head()




In [ ]:
final_df = house_crime.merge(
    deprivation_transport,
    on="LSOA code",
    how="left",
    validate="one_to_one"
)

final_df = final_df.drop(
    columns=["LA code", "LA name"]
)

In [ ]:
print(final_df.shape)
print(final_df.head())
print(final_df.columns)
output_path = DATA_DIR / "final_dataset.csv"

final_df.to_csv(output_path, index=False)

In [ ]:
print("Missing house prices:",
      final_df["average_price_2022"].isna().sum())

print("Missing crime:",
      final_df["total_crime_2022"].isna().sum())

print("Missing IMD:",
      final_df["IMD score"].isna().sum())

print("Missing PTAI:",
      final_df["AvPTAI2015"].isna().sum())